In [1]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point, shape
import ast

# Step 1: Load house data and convert to GeoDataFrame
houses_df = pd.read_csv("../data/housing_data_numeric_NLP.csv")

# Create Point geometry from longitude and latitude (note order: lon, lat)
houses_df['geometry'] = [Point(lon, lat) for lon, lat in zip(houses_df['longitude'], houses_df['latitude'])]
houses_gdf = gpd.GeoDataFrame(houses_df, geometry='geometry', crs="EPSG:4326")  # WGS 84

# Step 2: Load and convert neighborhood data
neigh_df = pd.read_csv("austin_neighborhood_boundaries.csv")

# Convert 'the_geom' strings to dictionaries safely
neigh_df['the_geom'] = neigh_df['the_geom'].apply(ast.literal_eval)

# Convert each dict into a Shapely geometry
neigh_df['geometry'] = neigh_df['the_geom'].apply(shape)
neigh_gdf = gpd.GeoDataFrame(neigh_df, geometry='geometry', crs="EPSG:4326")

# Step 3: Spatial join to add planning_area_name
joined = gpd.sjoin(houses_gdf, neigh_gdf[['planning_area_name', 'geometry']], how='left', predicate='within')

# Step 4: One-hot encode planning_area_name
joined_encoded = pd.get_dummies(joined, columns=['planning_area_name'])

# Step 5: Drop geometry and index_right (from spatial join), save to CSV
joined_encoded.drop(columns=['geometry', 'index_right'], errors='ignore').to_csv("../data/housing_data_with_neighborhoods_encoded.csv", index=False)
